# Weather Dashboard

A comprehensive weather dashboard that fetches real-time weather data from OpenWeatherMap API and provides visualizations.

## Features
- Real-time weather data for any city
- Current conditions, temperature, humidity, wind speed
- 5-day forecast visualization
- Weather pattern analysis
- Interactive plots and statistics

In [ ]:
# Install required libraries
import subprocess
import sys

packages = ['requests', 'pandas', 'matplotlib', 'seaborn', 'ipywidgets']
for package in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])

In [ ]:
import requests
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import json

# Set style
sns.set_style('darkgrid')
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 10

## Configuration

Get your free API key from [OpenWeatherMap](https://openweathermap.org/api)

In [ ]:
# Weather API Configuration
API_KEY = 'YOUR_OPENWEATHERMAP_API_KEY'  # Replace with your API key
BASE_URL = 'https://api.openweathermap.org/data/2.5'

# Default city for demonstration
DEFAULT_CITY = 'London'
DEFAULT_UNITS = 'metric'  # celsius, use 'imperial' for fahrenheit

## Weather API Functions

In [ ]:
def get_current_weather(city, api_key=API_KEY, units=DEFAULT_UNITS):
    """Fetch current weather data for a city"""
    try:
        url = f'{BASE_URL}/weather'
        params = {
            'q': city,
            'appid': api_key,
            'units': units
        }
        response = requests.get(url, params=params)
        response.raise_for_status()
        return response.json()
    except requests.exceptions.RequestException as e:
        print(f'Error fetching current weather: {e}')
        return None

def get_forecast(city, api_key=API_KEY, units=DEFAULT_UNITS):
    """Fetch 5-day forecast data for a city"""
    try:
        url = f'{BASE_URL}/forecast'
        params = {
            'q': city,
            'appid': api_key,
            'units': units
        }
        response = requests.get(url, params=params)
        response.raise_for_status()
        return response.json()
    except requests.exceptions.RequestException as e:
        print(f'Error fetching forecast: {e}')
        return None

## Data Processing Functions

In [ ]:
def parse_current_weather(data):
    """Parse current weather data into readable format"""
    if not data:
        return None
    
    return {
        'city': data['name'],
        'country': data['sys']['country'],
        'temperature': data['main']['temp'],
        'feels_like': data['main']['feels_like'],
        'temp_min': data['main']['temp_min'],
        'temp_max': data['main']['temp_max'],
        'pressure': data['main']['pressure'],
        'humidity': data['main']['humidity'],
        'wind_speed': data['wind']['speed'],
        'clouds': data['clouds']['all'],
        'description': data['weather'][0]['description'],
        'icon': data['weather'][0]['icon'],
        'timestamp': datetime.fromtimestamp(data['dt'])
    }

def parse_forecast(data):
    """Parse forecast data into DataFrame"""
    if not data:
        return None
    
    records = []
    for item in data['list']:
        records.append({
            'datetime': datetime.fromtimestamp(item['dt']),
            'temperature': item['main']['temp'],
            'feels_like': item['main']['feels_like'],
            'temp_min': item['main']['temp_min'],
            'temp_max': item['main']['temp_max'],
            'humidity': item['main']['humidity'],
            'pressure': item['main']['pressure'],
            'wind_speed': item['wind']['speed'],
            'clouds': item['clouds']['all'],
            'rain': item.get('rain', {}).get('3h', 0),
            'description': item['weather'][0]['description']
        })
    
    return pd.DataFrame(records)

## Display Functions

In [ ]:
def display_current_weather(weather_data):
    """Display current weather in a formatted way"""
    if not weather_data:
        print('No weather data available')
        return
    
    print('='*50)
    print(f"Current Weather in {weather_data['city']}, {weather_data['country']}")
    print('='*50)
    print(f"Time: {weather_data['timestamp'].strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"\nConditions: {weather_data['description'].title()}")
    print(f"Temperature: {weather_data['temperature']:.1f}°C (feels like {weather_data['feels_like']:.1f}°C)")
    print(f"Min/Max: {weather_data['temp_min']:.1f}°C / {weather_data['temp_max']:.1f}°C")
    print(f"\nHumidity: {weather_data['humidity']}%")
    print(f"Wind Speed: {weather_data['wind_speed']} m/s")
    print(f"Pressure: {weather_data['pressure']} hPa")
    print(f"Cloud Coverage: {weather_data['clouds']}%")
    print('='*50)

## Visualization Functions

In [ ]:
def plot_forecast(df, city):
    """Create forecast visualization plots"""
    if df is None or df.empty:
        print('No forecast data available')
        return
    
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    fig.suptitle(f'5-Day Weather Forecast for {city}', fontsize=16, fontweight='bold')
    
    # Temperature trend
    ax1 = axes[0, 0]
    ax1.plot(df['datetime'], df['temperature'], marker='o', linewidth=2, markersize=4, label='Actual')
    ax1.fill_between(df['datetime'], df['temp_min'], df['temp_max'], alpha=0.3, label='Min/Max Range')
    ax1.set_title('Temperature Trend')
    ax1.set_ylabel('Temperature (°C)')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    ax1.tick_params(axis='x', rotation=45)
    
    # Humidity
    ax2 = axes[0, 1]
    ax2.plot(df['datetime'], df['humidity'], marker='s', color='#3498db', linewidth=2, markersize=4)
    ax2.fill_between(df['datetime'], df['humidity'], alpha=0.3, color='#3498db')
    ax2.set_title('Humidity Level')
    ax2.set_ylabel('Humidity (%)')
    ax2.set_ylim(0, 100)
    ax2.grid(True, alpha=0.3)
    ax2.tick_params(axis='x', rotation=45)
    
    # Wind Speed
    ax3 = axes[1, 0]
    ax3.bar(df['datetime'], df['wind_speed'], color='#e74c3c', alpha=0.7)
    ax3.set_title('Wind Speed')
    ax3.set_ylabel('Wind Speed (m/s)')
    ax3.grid(True, alpha=0.3, axis='y')
    ax3.tick_params(axis='x', rotation=45)
    
    # Precipitation
    ax4 = axes[1, 1]
    ax4.bar(df['datetime'], df['rain'], color='#2ecc71', alpha=0.7)
    ax4.set_title('Precipitation')
    ax4.set_ylabel('Rain (mm/3h)')
    ax4.grid(True, alpha=0.3, axis='y')
    ax4.tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.show()

## Main Weather Dashboard Function

In [ ]:
def weather_dashboard(city=DEFAULT_CITY, api_key=API_KEY):
    """Main function to run the complete weather dashboard"""
    
    print(f'\nFetching weather data for {city}...')
    
    # Get current weather
    current_data = get_current_weather(city, api_key)
    if not current_data:
        print(f'Failed to fetch weather for {city}')
        return
    
    # Parse and display current weather
    weather_info = parse_current_weather(current_data)
    display_current_weather(weather_info)
    
    # Get and visualize forecast
    forecast_data = get_forecast(city, api_key)
    if forecast_data:
        forecast_df = parse_forecast(forecast_data)
        plot_forecast(forecast_df, city)
        
        # Print forecast statistics
        print('\nForecast Statistics (5 Days):')
        print(forecast_df[['temperature', 'humidity', 'wind_speed', 'rain']].describe().round(2))

## Usage Example

First, update the `API_KEY` variable with your OpenWeatherMap API key.

In [ ]:
# Run the weather dashboard
# Uncomment and run with your API key
# weather_dashboard('London')

## Multiple Cities Comparison

In [ ]:
def compare_cities(cities, api_key=API_KEY):
    """Compare weather across multiple cities"""
    results = []
    
    for city in cities:
        data = get_current_weather(city, api_key)
        if data:
            weather = parse_current_weather(data)
            results.append({
                'City': weather['city'],
                'Temperature (°C)': weather['temperature'],
                'Humidity (%)': weather['humidity'],
                'Wind Speed (m/s)': weather['wind_speed'],
                'Description': weather['description']
            })
    
    if results:
        df = pd.DataFrame(results)
        print('\nWeather Comparison:')
        print(df.to_string(index=False))
        return df
    return None

# Example: compare multiple cities
# cities = ['London', 'New York', 'Tokyo', 'Sydney']
# compare_cities(cities)

## Notes

- **API Key**: Get your free key from [OpenWeatherMap](https://openweathermap.org/api)
- **Rate Limits**: Free tier allows up to 60 calls/minute
- **Units**: Change to 'imperial' for Fahrenheit/mph
- **Error Handling**: The functions include error handling for network issues
- **Extensions**: You can add more features like alerts, historical comparison, or map visualization